In [ ]:
import yaml
from json_repair import repair_json

from openai import OpenAI
from tool_description_optimizer.common.client.llm_client import LLMClient
# 相关配置
from tool_description_optimizer.common.flow.flow_config import FlowConfig
from tool_description_optimizer.common.passk.prompt_registry import PromptRegistry
from tool_description_optimizer.src.optimizer.state import *

# 相关执行class
from tool_description_optimizer.src.optimizer.nn_recall_passk import recall_passk_function
try:
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
except ImportError as exc:  # pragma: no cover
    raise RuntimeError(
        "Missing langgraph dependencies. Please run: "
        "pip install langgraph langchain-core"
    ) from exc

from tool_description_optimizer.src.optimizer.llm_description_optimizer import LLMDescriptionOptimizer
from tool_description_optimizer.src.optimizer.llm_description_judge import LLMDescriptionJudge


/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/server/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:24<00:00, 16.47it/s]


In [29]:
from __future__ import annotations

from tool_description_optimizer.common.utils.utils import *

from tool_description_optimizer.src.optimizer.state import ToolOptimizerState

class LLMDescriptionOptimizer:
    @staticmethod
    def _gen_prompt(state: ToolOptimizerState, prompt: str, tools_dict: dict):
        best_record = state.get("best_record", None)
        if not best_record:
            return ""
        title = state.get("title", "")
        description = best_record.description
        top_case  = best_record.top_case
        miss_case = ""
        for k, v in top_case.items():
            if tools_dict.get(k, "") == "" or k == NO_CALL:
                miss_case += "\n- 未召回query集合: " + "、".join(v)
            else:
                miss_case += "\n[[负例样本详情]] \n- 未召回query集合：" + "、".join(v) + "\n- 上面的query召回其他的混淆工具描述如下：\n" + tools_dict[k]["description"]

        return prompt.replace("{{original_description}}", description).replace("{{case}}", miss_case).replace("{{title}}", title)

    @staticmethod
    def _vertify_result(response: dict):
        
        # check response result
        if len(set(response.keys()) - set(['thought', 'optimizer_description', 'change_reason', 'optimizer_keys', 'risk'])) > 0:
            return {}, False , "key error"
        # check optimizer_description 字符串
        optimizer_description = response["optimizer_description"]
        if len(optimizer_description) < 100:
            return {}, False, "optimizer_description error"
        
        # check change_reason 字符串
        change_reason = response["change_reason"]  #isinstance(raw_data, (dict, list))
        if isinstance(change_reason, list):
            change_reason = "".join(change_reason)
        
        # check optimizer_keys 字符串
        optimizer_keys = response['optimizer_keys']
        if isinstance(optimizer_keys, list):
            optimizer_keys = " ".join(optimizer_keys) 

        # check risk 字符串
        risk = response['risk']
        if isinstance(risk, list):
            risk = " ".join(risk) 
        return {
            'optimizer_description': optimizer_description, 
            'change_reason': change_reason, 
            'optimizer_keys': optimizer_keys, 
            'risk': risk
        }, True, ""




In [20]:


class ToolOptimizerGraph:
    def __init__(
        self,
        resource_id: str,
        llm_client: Optional[LLMClient] = None,
        prompt_path: str = "../config/prompts.json",
        flow_config_path: str = "../config/agent_config.yaml",
        tools_description_path: str = "../data/optimizer/tool_descriptions.json",
        test_data_path: str = "../data/optimizer/query.json",
        checkpointer: Optional[Any] = None,
    ) -> None:
        # 设置通用的 参数
        self.prompts = PromptRegistry(prompt_path).get_promts()
        self.flow_config = FlowConfig(flow_config_path)
        self.checkpointer = checkpointer or InMemorySaver()
        self.last_tools_description_path = tools_description_path
        # 设置 tool resource_id
        self.resource_id = resource_id

        # 加载tools 这个列表
        self.tools_dict = load_tools_from_json(self.last_tools_description_path)

        # 加载测试集
        # test_data_path: str = "../data/optimizer/query.json"
        # load_tools_from_json(test_data_path)#
        self.query_good_all_dict = load_tools_from_json(test_data_path)
        self.query_good_dict = self.query_good_all_dict[resource_id]

        # node -> model
        self.node_model_map: Dict[str, str] = {} 
        if self.flow_config.node_model_map:
            self.node_model_map.update({str(k): str(v) for k, v in self.flow_config.node_model_map.items()})

        # node -> temperature
        self.node_temperature_map: Dict[str, float] =  {} 
        if self.flow_config.node_temperature_map:
            self.node_temperature_map.update({str(k): float(v) for k, v in self.flow_config.node_temperature_map.items()})
 
        llm_dict = {}
        if len(self.flow_config.clients) != 0:
            for k, v in self.flow_config.clients.items():
                if "tianchi" in v["base_url"]:
                    v['host'] = "tianchi-proxy.baidu-int.com"
                    v['appid'] = 'app-CdjpA4YQ'
                    llm_dict[k] = LLMClient(**v)

        # llm_client
        if llm_client:
            self.llm_client = llm_client
        else:
            self.llm_client = LLMClient(**self.flow_config.config["default_client"])

        # node_llm_client_map
        self.node_llm_clients: Dict[str, LLMClient] = {}
        if self.flow_config.node_llm_client_map:
            self.node_llm_clients.update({
                str(k): llm_dict[v] for k, v in self.flow_config.node_llm_client_map.items()
            })

    # ---------- generate text ----------

    # node -> client
    def _client_for(self, node_name: str) -> LLMClient:
        return self.node_llm_clients.get(node_name, self.llm_client)

    # node -> model name
    def _model_for(self, node_name: str) -> str:
        return self.node_model_map.get(node_name, "deepseek-v3.2")
    # node -> temperature 
    def _temperature_for(self, node_name: str) -> float:
        return self.node_temperature_map.get(node_name, 0.8)

    def _generate_text(
        self,
        node_name: str,
        prompt: str,
        max_tokens: int = 2048,
        extra_body: Dict[str, Any] = {}
    ) -> Any:
        client = self._client_for(node_name)
        response =  client.generate_text(
            prompt = prompt,
            model=self._model_for(node_name),
            temperature=self._temperature_for(node_name),
            extra_body = extra_body
        )
        return client.parse_chat_content(response)
    
    # --------- 统计和check工具 ----------------------
    def statistic_check_tool(self, 
                    state: ToolOptimizerState):
        """示例主函数：你可以替换为自己的 JSON 文件路径。"""
        best_record = state.get("best_record", None)
        if best_record is not None:
            self.last_tools_description_path = best_record.tool_path

        self.tools_dict = load_tools_from_json(self.last_tools_description_path) 
        statistic_check_version = "version_" + str(state.get("current_version_id", 0))
        print("当前执行的版本：", statistic_check_version)
        statistic_output = "../recall_outputs/optimizer/" + statistic_check_version + "/" + self.resource_id
    
        detaildf = recall_passk_function(self.tools_dict, self.query_good_dict, self.resource_id, statistic_output)

        # 计算 top 1 的 precision
        detaildf_top1 = detaildf[(detaildf['k']==1) & (detaildf['view']=="merged")]
        relevants = detaildf_top1['gold_ids'].to_list()
        retrieveds = detaildf_top1['recall_ids'].to_list()
        precision1 = precision_at_k_batch(retrieveds, relevants, 1)
        recall1 = recall_at_k_batch(retrieveds, relevants, 1)

        # 计算 top 3 的 precision
        detaildf_top3 = detaildf[(detaildf['k']==3) & (detaildf['view']=="merged")]
        relevants = detaildf_top3['gold_ids'].to_list()
        retrieveds = detaildf_top3['recall_ids'].to_list()
        precision3 = precision_at_k_batch(retrieveds, relevants, 3)
        recall3 = recall_at_k_batch(retrieveds, relevants, 3)

        # 统计 top 3 的结果
        tools_query = {}
        tools_case_ids = []
        for indx, row in detaildf_top3.iterrows():
            recall_ids = row["recall_ids"]
            if len(recall_ids) == 0:
                recall_ids = [NO_CALL] 
            if len(tools_query.get(recall_ids[0], [])) == 0:
                tools_query[recall_ids[0]] =  []
            tools_query[recall_ids[0]].append(row["query"])
            tools_case_ids.append(recall_ids[0])
        top_tools_case = {k: tools_query[k] for k in get_k_tool(tools_case_ids, 3)}
        tools_case_all = {k: tools_query[k] for k in set(tools_case_ids)}

        version_id, next_version_id = next_version_pair(state)

        versionInfo = VersionRecord(
            version_id = statistic_check_version, 
            parent_version_id = version_id,
            stage = "statistic",
            description = self.tools_dict[self.resource_id]["description"],
            case_result = tools_case_all,
            top_case = top_tools_case,
            tool_path = statistic_output + "/tool_prompt.json",
            recall1 = recall1,
            precision1 = precision1,
            recall3 = recall3,
            precision3 = precision3
        )
        best_record = state.get("best_record", None)
        
        # 判断是否需要更新最佳记录
        # 条件1: best_record 不存在 (即为 None)
        # 条件2: 新的指标 (recall3, precision3) 优于旧的 best_record
        should_update = (best_record is None) or \
                (recall3 > best_record.recall3 and precision3 > best_record.precision3)

        with open(statistic_output + "/tool_prompt.json","w") as w:
            json.dump(self.tools_dict, w, ensure_ascii=False, indent=2)

        if should_update:
            return {
                    "current_version_id": version_id,
                    "next_version_id": next_version_id,
                    "version_history": append_version_history(state, versionInfo),
                    "best_description": state.get("current_description", ""),
                    "best_version_id": state.get("current_version_id", 0),
                    "best_record": versionInfo,

                }
        else:
            return {
                "current_version_id": version_id,
                "next_version_id": next_version_id,
                "version_history": append_version_history(state, versionInfo)
            }


    # ---------  LLMNodeOptimizer node--------------
    def llm_description_optimizer(self, 
                                  state: ToolOptimizerState):
        prompt = LLMDescriptionOptimizer._gen_prompt(state, self.prompts['optimizer'], self.tools_dict)
        for _ in range(3):
            response, stype, flag = self._generate_text(node_name = "optimizer", prompt = prompt)
            response, flag, error_type = LLMDescriptionOptimizer._vertify_result(response)
            if flag:
                break
        if flag:
            optimizer_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "optimizer",
                info = response
            )
            return {
                "optimizer_history": append_version_history(state, optimizer_record)
            }

    # ---------  LLMNodeCritic node--------------
    def llm_description_judge(self,  state: ToolOptimizerState):
        prompt = LLMDescriptionJudge._gen_prompt(state, self.prompts['judge'])

        for _ in range(3):
            response, stype, flag = self._generate_text(node_name = "judge", prompt = prompt)
            response, flag, error_type = LLMDescriptionJudge._vertify_result(response)
            if flag:
                break
        if flag:
            relevance_score = response["relevance_score"]
            if relevance_score == 3:
                self.tools_dict[self.resource_id]['description'] = optimizer_description

            judge_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "judge",
                info = response
            )
            return {
                "judge_history": append_judge_history(state, judge_record)
            }


    # ---------- graph build ----------
def should_continue(state: ToolOptimizerState) -> Literal["description_optimizer", END]:
    # 达到最大轮次
    max_iteration = state.get("max_iterations", 3)
    if state.get("current_version_id", 10) > max_iteration:
        return END
    return "description_optimizer"

query_gold_dict = load_tools_from_json("../data/optimizer/query.json")

#------- init ----
resource_id = '5868'
dag = ToolOptimizerGraph(resource_id = resource_id,
                         prompt_path ="../../config/prompts.json",
                         flow_config_path ="../../config/agent_config.yaml",
                         tools_description_path = "../data/optimizer/tool_descriptions.json",
                         test_data_path  = "../data/optimizer/query.json",
                         checkpointer = None)

graph = StateGraph(ToolOptimizerState)
graph.add_node("statistic_check_tool", dag.statistic_check_tool)
graph.add_node("description_optimizer", dag.llm_description_optimizer)
graph.add_node("description_judge", dag.llm_description_judge)

graph.set_entry_point("statistic_check_tool")
graph.add_conditional_edges(
    "statistic_check_tool",
    should_continue,
    {
        "description_optimizer": "description_optimizer",
        END: END
    }
)
graph.add_edge("statistic_check_tool", "description_optimizer")
graph.add_edge("description_optimizer", END)
# graph.add_edge("description_judge", END)

compiled_graph = graph.compile()

initial_state = {
        "resource_id": resource_id,
        "title": dag.tools_dict[resource_id]["title"],
        "original_description": dag.tools_dict[resource_id]["description"],
        # 当前工作基线指针（可手动/自动回滚）
        "current_version_id": 0,
        "current_description": dag.tools_dict[resource_id]["description"],
        # 下一个版本的id
        "next_version_id": 1,
        # 固定参数
        "max_iterations": 3,
        "iteration": 0,
        # 版本历史仓库：全量快照存储
        "version_history": [],
        #记录历史
        "optimizer_history": [],
        "critic_history": [],
        "generate_history": [],
        "judge_history": [],
        "verify_judge_history": [],
    }
state = compiled_graph.invoke(initial_state)
save_pickle(state, "../recall_outputs/summary/" + resource_id + ".pkl")

当前执行的版本： version_0
开始建库: description
Building embeddings for view=description, size=362
开始建库: title_description
Building embeddings for view=title_description, size=362
Total tools indexed: 362
Total eval queries: 7
召回结果保存完成：
summary_csv: ../recall_outputs/optimizer/version_0/5868/tool_pass_at_k_recall_summary.csv
detail_csv: ../recall_outputs/optimizer/version_0/5868/tool_pass_at_k_recall_detail.csv
summary_jsonl: ../recall_outputs/optimizer/version_0/5868/tool_pass_at_k_recall_summary.jsonl
detail_jsonl: ../recall_outputs/optimizer/version_0/5868/tool_pass_at_k_recall_detail.jsonl
excel: ../recall_outputs/optimizer/version_0/5868/tool_pass_at_k_recall.xlsx


In [28]:
response

{'semantic_analysis': '详细对比原始描述与优化后描述：\n- 原始描述的核心意图：描述一个用于游戏场景、提供精准/泛化游戏下载资源的工具，明确其返回的内容项、组件样式和交互逻辑。\n- 优化后描述的核心意图：在原始描述基础上，进一步明确了工具的应用场景（如游戏平台、商店），并举例说明了查询示例（如“taptap”、“steam”），同时完整保留了原始描述的全部核心语义。\n- 两者在核心主题（游戏下载资源展示）、关键对象（游戏信息、组件）、核心目标（支持用户查找和下载游戏）和表达意图上完全一致。\n- 优化后未遗漏原始描述中的任何关键语义，反而对适用场景进行了合理且具体的补充说明。\n- 优化后未引入与原始描述不一致的新语义，所有新增内容均为对原始意图的合理解释和场景延伸，未改变原意。\n- 整体语义匹配程度极高。\n\n语义匹配分数：98/100',
 'scene_comparison': '详细对比原始描述与优化后描述的适用场景：\n- 原始描述适用于：游戏场景，具体指需要返回精准或泛化游戏下载资源的场合。\n- 优化后描述适用于：游戏场景，并具体化为游戏平台、游戏商店、应用市场等，且举例了“taptap”、“steam”等查询场景。\n- 两者的用户对象（寻找游戏下载资源的用户）、使用环境（数字平台）、业务场景（游戏信息展示与分发）和问题背景完全一致。\n- 优化后未改变原始描述的使用场景，而是在原始较宽泛的“游戏场景”基础上，进行了合理、具体的场景举例和细化，这属于有益的澄清和扩展，并未偏离核心。\n- 优化后增加的新场景（如具体平台名称举例）是原始“游戏场景”的自然子集和具体化，并非不合理的新场景。\n\n场景匹配分数：95/100',
 'function_comparison': '详细对比原始描述与优化后描述的核心功能：\n- 原始描述提供的功能：1) 返回包含游戏名、icon、四要素等在内的游戏信息；2) 按照指定样式展示这些信息组件；3) 支持点击各组件跳转详情页、点击隐私/权限跳转对应页、点击下载按钮触发下载并跳转详情页的交互。\n- 优化后描述提供的功能：完全覆盖了原始描述的所有功能点，包括返回内容、组件样式和全部交互逻辑，表述更为连贯。\n- 核心功能目标（提供游戏信息展示与下载引导）完全一致。\n- 功能范围未发生

In [30]:
from __future__ import annotations

from typing import List

from tool_description_optimizer.src.optimizer.state import ToolOptimizerState

class LLMDescriptionJudge:
    @staticmethod
    def _gen_prompt(state: ToolOptimizerState, prompt: str):
        optimizer_history = state.get("optimizer_history", [])
        if len(optimizer_history) == 0:
            return ""
        infoRecord = optimizer_history[-1]
        original_description = state.get("original_description", "")
        best_description = infoRecord.info.get("optimizer_description", "")
        if len(original_description) < 10 or len(best_description) < 10:
            return ""
        return prompt.replace("{{before_text}}", original_description).replace("{{after_text}}", best_description), best_description

    @staticmethod
    def _vertify_result(response: dict):
        
        # check response result
        if len(set(response.keys()) - set(['semantic_analysis', 'scene_comparison', 'function_comparison', 'content_quality', 'relevance_reason', 'relevance_score'])) > 0:
            return {}, False , "key error"
        # check semantic_analysis 字符串
        semantic_analysis = response["semantic_analysis"]
        if len(optimizer_description) < 10:
            return {}, False, "semantic_analysis error"
        
        # check scene_comparison 字符串
        scene_comparison = response["scene_comparison"]  
        if len(scene_comparison) < 10:
            return {}, False, "scene_comparison error"
        
        # check function_comparison 字符串
        function_comparison = response["function_comparison"]  
        if len(function_comparison) < 10:
            return {}, False, "function_comparison error"

        # check content_quality 字符串
        content_quality = response["content_quality"]  
        if len(content_quality) < 10:
            return {}, False, "content_quality error"

        # check relevance_reason 字符串
        relevance_reason = response["relevance_reason"]  
        if len(relevance_reason) < 10:
            return {}, False, "relevance_reason error"

        # check relevance_score 类别
        relevance_score = response['relevance_score']
        if isinstance(relevance_score, (float, str)):
            relevance_score = int(relevance_score) 
        return {
            'semantic_analysis': semantic_analysis, 
            'scene_comparison': scene_comparison, 
            'function_comparison': function_comparison, 
            'content_quality':content_quality,
            'relevance_reason': relevance_reason,
            'relevance_score': relevance_score
        }, True, ""




In [103]:
def pass_at_k(n: int, c: int, k: int) -> float:
    """
    计算 Pass@K (通常用于代码生成或生成式检索评估)
    Args:
        n: 总生成样本数
        c: 正确样本数
        k: 评估的截断阈值 (Top-K)
    Returns:
        Pass@K 得分 (0.0 ~ 1.0)
    """
    if n - c < k:
        return 1.0
    # 使用组合数公式计算至少命中一个正确结果的概率
    # 1 - C(n-c, k) / C(n, k)
    # 为避免阶乘溢出，使用连乘计算
    prob_no_pass = 1.0
    for i in range(k):
        prob_no_pass *= (n - c - i) / (n - i)
    return 1.0 - prob_no_pass


def precision_at_k(retrieved: List[Union[str, int]], relevant: List[Union[str, int]], k: int) -> float:
    """
    计算 Precision@K (Top-K 结果中相关文档的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Precision@K 得分 (0.0 ~ 1.0)
    """
    if k <= 0:
        return 0.0
    # 截取前 k 个结果
    top_k_results = retrieved[:k]
    if not top_k_results:
        return 0.0
    
    # 计算前 k 个结果中有多少是相关的
    relevant_count = sum(1 for item in top_k_results if item in relevant)
    return relevant_count / len(relevant)

def precision_at_k_batch(retrieveds: List[List[Union[str, int]]], relevants: List[List[Union[str, int]]], k: int) -> float:
    """
    计算 Precision@K (Top-K 结果中相关文档的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Precision@K 平均得分 (0.0 ~ 1.0)
    """
    
    if len(relevants) != len(retrieveds):
        return 0.0
    else:
        precision_scores = []
        for retrieved, relevant in zip(retrieveds, relevants):
            precision_scores.append(precision_at_k(retrieved, relevant, k))
    print("precision_scores", precision_scores)
    return sum(precision_scores) / len(retrieveds)



def recall_at_k(retrieved: List[Union[str, int]], relevant: List[Union[str, int]], k: int) -> float:
    """
    计算 Recall@K (所有相关文档中，被检索到且排在 Top-K 的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Recall@K 得分 (0.0 ~ 1.0)
    """
    if not relevant:
        return 0.0
    
    # 截取前 k 个结果
    top_k_results = retrieved[:k]
    
    # 计算前 k 个结果中命中了多少个真实相关文档
    relevant_found = sum(1 for item in top_k_results if item in item in relevant)
    return relevant_found / len(relevant)

def recall_at_k_batch(retrieveds: List[List[Union[str, int]]], relevants: List[List[Union[str, int]]], k: int) -> float:
    """
    计算 Recall@K (所有相关文档中，被检索到且排在 Top-K 的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Recall@K 平均得分 (0.0 ~ 1.0)
    """
    if len(relevants) != len(retrieveds):
        return 0.0
    else:
        recall_scores = []
        for retrieved, relevant in zip(retrieveds, relevants):
            recall_scores.append(recall_at_k(retrieved, relevant, k))
    print("recall_scores", recall_scores)
    return sum(recall_scores) / len(retrieveds)


In [111]:

import json
import time
from typing import Any, Dict, List, Optional
# from llm_client import LLMClient
from openai import OpenAI

class LLMClient:
    """OpenAI-compatible chat-completions client."""
    def __init__(self, base_url, api_key, timeout, max_retry, host = None, appid = None):
        self.base_url = base_url
        self.api_key = api_key
        self.max_retry=3
        self.sleep_seconds=3
        default_headers = {}
        if not host:
            default_headers["Host"] = host
        if not appid:
            default_headers["appid"] = appid
        if len(default_headers) == 0:
            self.client = OpenAI(
                base_url=base_url,
                api_key=api_key,
                timeout=timeout,
                max_retries=max_retry,  # 这里用我们自己的重试逻辑，避免 SDK + 手动双重重试
                default_headers= default_headers
            )
        else:
            self.client = OpenAI(
                base_url=base_url,
                api_key=api_key,
                timeout=timeout,
                max_retries=max_retry,  # 这里用我们自己的重试逻辑，避免 SDK + 手动双重重试
    #             default_headers= default_headers
            )
 
    def fetch_response(self,
            query: str,
            doc: str,
            prompt: str = None,
            messages: Optional[List[Dict[str, str]]] = None,
            model: Optional[str] = None,
            temperature: Optional[float] = None,
            max_tokens: Optional[int] = None,
            extra_body: Optional[Dict[str, Any]] = None,
        ) -> Dict[str, Any]:

        if not messages:
            messages = [{"role": "user",
                        "content": prompt}]
            
        last_error = None
        request_params = {
            "model": model,
            "messages": messages,
            "temperature": temperature if temperature is not None else 0.8,
            "max_tokens": max_tokens if max_tokens is not None else 8196,
        }
        if extra_body:
            request_params["extra_body"] = extra_body
        
        print("request_params", request_params)
        print("-" * 100)
        print()
        self.client.chat.completions.create(**request_params)
        
        for retry_idx in range(self.max_retry):
            try:
                response = self.client.chat.completions.create(**request_params)

                # OpenAI SDK 返回的是对象，需要转成 dict，方便 json.dumps
                output_json = response.model_dump(mode="json")
                return output_json

            except Exception as e:
                last_error = e
                print(f"[Retry {retry_idx + 1}/{self.max_retry}] 请求失败: {e}")
                time.sleep(self.sleep_seconds)

        raise RuntimeError(f"请求重试 {self.max_retry} 次后仍失败: {last_error}")

#     def request_model(
#         self,
#         *,
#         prompt: Optional[str] = None,
#         messages: Optional[List[Dict[str, str]]] = None,
#         model: Optional[str] = None,
#         temperature: Optional[float] = None,
#         max_tokens: Optional[int] = None,
#         extra_body: Optional[Dict[str, Any]] = None,
#     ) -> Dict[str, Any]:
#         """Use requests to call an OpenAI-compatible chat-completions model API.

#         This is the lowest-level model access method. All LangGraph nodes can
#         share it, and you can also call it directly when debugging a model
#         service such as vLLM, SGLang, LMDeploy, or OpenAI-compatible gateways.

#         Args:
#             prompt: Simple user prompt. Used when `messages` is not provided.
#             messages: Full chat messages, e.g.
#                 [{"role": "system", "content": "..."},
#                  {"role": "user", "content": "..."}]
#             model: Override model name for this request.
#             temperature: Override sampling temperature.
#             max_tokens: Optional output token limit.
#             extra_body: Extra OpenAI-compatible fields, such as top_p, stop,
#                 repetition_penalty, response_format, etc.

#         Returns:
#             Raw JSON response from /chat/completions.
#         """
#         if not self.base_url:
#             raise RuntimeError("OPENAI_API_BASE is empty. Please set it first.")

#         url = self.base_url.rstrip("/")
#         if not url.endswith("/chat/completions"):
#             url = url + "/chat/completions"

#         headers = {"Content-Type": "application/json"}
#         if self.api_key:
#             headers["Authorization"] = f"Bearer {self.api_key}"

#         if messages is None:
#             messages = [{"role": "user", "content": prompt or ""}]

#         payload: Dict[str, Any] = {
#             "model": model or self.model,
#             "messages": messages,
#             "temperature": 0.2 if temperature is None else temperature,
#         }
#         if max_tokens is not None:
#             payload["max_tokens"] = max_tokens
#         if extra_body:
#             payload.update(extra_body)

#         last_error: Optional[Exception] = None
#         for attempt in range(1, self.max_retry + 1):
#             try:
#                 resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
#                 resp.raise_for_status()
#                 return resp.json()
#             except Exception as exc:  # noqa: BLE001
#                 last_error = exc
#                 if attempt < self.max_retry:
#                     time.sleep(1)

#         raise RuntimeError(f"LLM request failed after {self.max_retry} retries: {last_error}")

    def parse_chat_content(self, response: Dict[str, Any]) -> str:
        """Extract assistant text from an OpenAI-compatible response."""
        try:
            return str(response["choices"][0]["message"]["content"])
        except (KeyError, IndexError, TypeError) as exc:
            raise ValueError(f"Invalid chat completion response: {response}") from exc

    def generate_text(
        self,
        query: str,
        doc: str,
        prompt: str = None,
        *,
        messages: Optional[List[Dict[str, str]]] = None,
        model: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        extra_body: Optional[Dict[str, Any]] = None,
    ) -> str:
        """Call model and return only assistant content.

        `model` and `temperature` can be overridden per node, so a LangGraph
        graph can use lightweight models for routing and stronger models for
        planning/writing/critique. Internally this delegates to `request_model`,
        the shared requests-based access method.
        """
        response = self.fetch_response(
            query = query,
            doc = doc,
            prompt = prompt,
            model=model,
            messages = messages,
            temperature = temperature,
            max_tokens = max_tokens,
            extra_body = extra_body,
        )
        return {
                "query": query,
                "doc": doc,
                "request_data": self.parse_chat_content(response),
                }
if __name__ == "__main__":

    llmclient = LLMClient(
        base_url="http://10.215.195.160:8880/v1",
        api_key="zacharychu",
        timeout=60,
        max_retry=3,
    )
    
    response = llmclient.generate_text(
        query = "nihao",
        model = "Qwen3-8B",
        doc = "生成",
        messages=[
            {"role": "system", "content": "你是一个严谨、简洁的中文助手。"},
        ],
        temperature=0.2,
        max_tokens=1024,
        # extra_body 可以放 top_p、stop、response_format 等 OpenAI-compatible 参数。
        extra_body={"top_p": 0.9},
    )

    print("RAW RESPONSE:")
    print(json.dumps(response, ensure_ascii=False, indent=2))
    print("\nCONTENT:")

request_params {'model': 'Qwen3-8B', 'messages': [{'role': 'system', 'content': '你是一个严谨、简洁的中文助手。'}], 'temperature': 0.2, 'max_tokens': 1024, 'extra_body': {'top_p': 0.9}}
----------------------------------------------------------------------------------------------------

RAW RESPONSE:
{
  "query": "nihao",
  "doc": "生成",
  "request_data": "好的，我将以严谨、简洁的方式为您提供帮助。请直接提出您的问题或需求。"
}

CONTENT:
